### Measurement-Bias Correction — EC-NAS (RQ3)

EC-NAS's energy is measured via **Carbontracker**, a software-based tool that reads chip-level power sensors rather than an external hardware watt-meter (unlike BUTTER-E, which uses node-level watt-meters — the ground-truth measurement type). Per Fischer et al. (2025), Carbontracker underestimates true energy consumption relative to hardware watt-meter ground truth by roughly **20-30%**.

**Correction applied:** `target_corrected = target × 1.25` — the midpoint of the 20-30% range (`1 / (1 - 0.25) ≈ 1.333` would be the multiplicative inverse of a 25% underestimate; `1.25` is used here as a simpler, directly-stated midpoint multiplier per the instruction, not a precision-derived inverse — see the note below on what this number actually represents).

**This is explicitly an approximation, not a per-architecture correction, and that limitation is worth stating plainly:**
- The true underestimation almost certainly varies by architecture, GPU, and workload characteristics (idle power draw, sampling rate relative to run duration, etc.) — a single scalar multiplier cannot capture that variation, it only shifts the whole distribution's level.
- `1.25×` is a plain multiplier on the reported value, not a bias-correction derived from `1/(1-underestimate)` (which would give ~1.25-1.43× depending on whether "20-30% underestimate" means the measured value is 70-80% of truth, or 20-30% below truth in some other sense). Since Fischer et al.'s own precise definition isn't re-derived here, `1.25` is used as stated — simple, documented, and clearly flagged as approximate rather than precise.
- **Flag on internal consistency, not resolved here:** `01a_butter_e_eda.ipynb`'s own earlier text cites Carbontracker underestimation as "up to 40%" (also attributed to Fischer, 2025), which doesn't match the "20-30%" figure used for this correction. Both figures trace to the same cited source in this project's own notes; which one is the paper's actual reported range isn't re-verified from the primary source here — flagged so it can be reconciled against the actual Fischer et al. (2025) text before this correction is treated as final for the thesis write-up.

In [ ]:
# LOAD & APPLY CORRECTION

import pandas as pd

CORRECTION_FACTOR = 1.25  # midpoint of Fischer et al. (2025)'s 20-30% Carbontracker underestimation range

df = pd.read_csv("../../data/processed/ec_nas/ec_nas_features.csv")

df["target_corrected"] = df["target"] * CORRECTION_FACTOR

OUT_PATH = "../../data/processed/ec_nas/ec_nas_features.csv"
df.to_csv(OUT_PATH, index=False)

print(df.shape)
print(df[["target", "target_corrected"]].describe())

In [ ]:
# SANITY CHECK

print(df.shape)
print(df.columns.tolist())
print("nulls:", df.isnull().sum().sum())
df[["target", "target_corrected"]].head()